# Tome 1, Chapter 6: Quadratic Reciprocity (Python companion)

Exercises first, solutions at the end. Computer-science threads: the **Jacobi symbol algorithm** (reciprocity as a
Euclid-like algorithm), **Tonelli–Shanks** square roots modulo $p$, and the **Solovay–Strassen** primality test.

In [ ]:
import math
import random
import numpy as np
import matplotlib.pyplot as plt
from sympy import primerange, isprime, legendre_symbol, randprime

## Exercises

**P6.1 (★) Reciprocity, verified.** Check Theorem 6.5 for all pairs of distinct odd primes below 500, computing Legendre symbols with Euler's criterion. Then draw Eisenstein's lattice-point picture for $p = 11$, $q = 7$.

In [ ]:
# Your code here


**P6.2 (★★) CS application: the Jacobi symbol in $O(\log n)$.** Implement `jacobi(a, n)` using only: reduce $a$ mod $n$; pull out factors of 2 with $\left(\frac2n\right)$; swap with reciprocity. Compare with SymPy on random 500-bit inputs, and count the steps.

In [ ]:
# Your code here


**P6.3 (★★) CS application: Tonelli–Shanks.** Write $p - 1 = 2^s q$ with $q$ odd. Given a residue $a$, find $x$ with $x^2 \equiv a$: start from $x = a^{(q+1)/2}$, $t = a^q$, and a non-residue $z$ ($c = z^q$), and repeatedly fix the 2-power part of $t$. Implement it and test on 1000 random residues modulo a 256-bit prime. (For $p \equiv 3 \pmod 4$, $x = a^{(p+1)/4}$ works directly. Why?)

In [ ]:
# Your code here


**P6.4 (★★) CS application: Solovay–Strassen.** For odd $n$, if $n$ is prime then $a^{(n-1)/2} \equiv \left(\frac an\right)$ for all $a$. For composite $n$, at least half the bases $a$ violate this. Implement the test, run it on the Carmichael numbers of Tome 0 (which fool the Fermat test), and estimate the fraction of liars for a few composites.

In [ ]:
# Your code here


**P6.5 (★★) Pépin and Fermat numbers.** Use Pépin's test (Exercise 6.6) to show that $\mathcal F_1, \dots, \mathcal F_4$ are prime and $\mathcal F_5, \dots, \mathcal F_{12}$ are composite. $\mathcal F_{12}$ has 1234 digits: the test is quick even though nobody knows its full factorisation.

In [ ]:
# Your code here


**P6.6 (★★★) Patterns of residues.** For $p = 1009$ plot the quadratic residues as black/white stripes. Count runs of consecutive residues and compare with Exercise 6.5. Then compare $\sum_{n\le x} \left(\frac np\right)$ (a random-looking walk) with the Pólya–Vinogradov bound $\sqrt p\ln p$.

In [ ]:
# Your code here


---
## Solutions

### P6.1

In [ ]:
def legendre(a, p):
    r = pow(a, (p - 1) // 2, p)
    return -1 if r == p - 1 else r

ps = list(primerange(3, 500))
for p in ps:
    for q in ps:
        if p < q:
            assert legendre(p, q) * legendre(q, p) == (-1) ** ((p - 1) // 2 * (q - 1) // 2)
print(f"reciprocity verified for {len(ps) * (len(ps) - 1) // 2} pairs")

p, q = 11, 7
pts = [(x, y) for x in range(1, (p - 1) // 2 + 1) for y in range(1, (q - 1) // 2 + 1)]
below = [(x, y) for x, y in pts if q * x > p * y]
plt.figure(figsize=(5, 4))
plt.scatter(*zip(*pts), c=["tab:blue" if pt in below else "tab:orange" for pt in pts], s=80)
t = np.linspace(0, p / 2, 10); plt.plot(t, q * t / p, "k-")
plt.title(f"T(q,p) = {len(below)} below, T(p,q) = {len(pts) - len(below)} above, total {len(pts)}"); plt.show()

### P6.2

In [ ]:
def jacobi(a, n):
    assert n > 0 and n % 2 == 1
    a %= n
    result, steps = 1, 0
    while a:
        while a % 2 == 0:
            a //= 2
            if n % 8 in (3, 5):
                result = -result
        a, n = n, a                     # reciprocity
        if a % 4 == 3 and n % 4 == 3:
            result = -result
        a %= n
        steps += 1
    return (result if n == 1 else 0), steps

from sympy import jacobi_symbol
for _ in range(2000):
    n = random.getrandbits(500) | 1
    a = random.getrandbits(500)
    assert jacobi(a, n)[0] == jacobi_symbol(a, n)
print("jacobi() agrees with SymPy; steps for a 500-bit input:", jacobi(random.getrandbits(500), random.getrandbits(500) | 1)[1])
print("(219 | 383) =", jacobi(219, 383)[0])

### P6.3

In [ ]:
def tonelli_shanks(a, p):
    assert legendre(a, p) == 1
    if p % 4 == 3:
        return pow(a, (p + 1) // 4, p)
    q, s = p - 1, 0
    while q % 2 == 0:
        q //= 2; s += 1
    z = next(z for z in range(2, p) if legendre(z, p) == -1)
    m, c, t, x = s, pow(z, q, p), pow(a, q, p), pow(a, (q + 1) // 2, p)
    while t != 1:
        i, t2 = 0, t                    # least i with t^(2^i) = 1
        while t2 != 1:
            t2 = t2 * t2 % p; i += 1
        b = pow(c, 1 << (m - i - 1), p)
        m, c, t, x = i, b * b % p, t * b * b % p, x * b % p
    return x

for p in [randprime(2**255, 2**256) for _ in range(3)]:
    for _ in range(1000):
        a = pow(random.randrange(1, p), 2, p)
        x = tonelli_shanks(a, p)
        assert x * x % p == a
print("Tonelli-Shanks: 3000 square roots modulo 256-bit primes, all correct")

For $p \equiv 3 \pmod 4$: $x^2 = a^{(p+1)/2} = a \cdot a^{(p-1)/2} = a\left(\frac ap\right) = a$. The invariant of Tonelli–Shanks is $x^2 \equiv at$ with $t$ of order dividing $2^{m-1}$; each step strictly lowers the order of $t$, so it ends with $t = 1$.

### P6.4

In [ ]:
def solovay_strassen(n, rounds=20):
    if n < 5 or n % 2 == 0:
        return n in (2, 3)
    for _ in range(rounds):
        a = random.randrange(2, n - 1)
        j = jacobi(a, n)[0]
        if j == 0 or pow(a, (n - 1) // 2, n) != j % n:
            return False
    return True

carmichael = [561, 1105, 1729, 2465, 2821, 6601, 8911, 10585, 15841, 29341]
print("Carmichael numbers declared prime:", [n for n in carmichael if solovay_strassen(n)])
for n in [561, 1105, 1729, 9, 91, 341]:
    liars = sum(1 for a in range(1, n) if math.gcd(a, n) == 1 and pow(a, (n - 1) // 2, n) == jacobi(a, n)[0] % n)
    print(f"n = {n}: Euler liars {liars} / {n - 1} = {liars / (n - 1):.3f}")
assert all(solovay_strassen(p) for p in primerange(3, 5000))

The set of liars is a proper subgroup of $(\mathbb Z/n\mathbb Z)^\times$ when $n$ is composite, hence has at most half its size (Lagrange's theorem, Tome 2). After $k$ rounds, a composite survives with probability at most $2^{-k}$.

### P6.5

In [ ]:
for n in range(1, 13):
    F = 2 ** (2**n) + 1
    print(f"F_{n} ({len(str(F))} digits):", "prime" if pow(3, (F - 1) // 2, F) == F - 1 else "composite")

### P6.6

In [ ]:
p = 1009
chi = np.array([legendre(a, p) for a in range(1, p)])
plt.figure(figsize=(12, 1.2)); plt.imshow(chi[None, :200] > 0, cmap="Greys_r", aspect="auto"); plt.yticks([]); plt.title("quadratic residues mod 1009 (white), a = 1..200"); plt.show()
both = sum(1 for a in range(1, p - 1) if chi[a - 1] == 1 and chi[a] == 1)
print("pairs (a, a+1) both residues:", both, "  formula:", (p - 4 - legendre(-1, p)) // 4)
walk = np.cumsum(chi)
plt.plot(walk); plt.axhline(math.sqrt(p) * math.log(p), ls="--", c="r"); plt.axhline(-math.sqrt(p) * math.log(p), ls="--", c="r")
plt.title("partial sums of the Legendre symbol mod 1009 and the Polya-Vinogradov bound"); plt.show()